In [ ]:
!pip install tensorflow numpy

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

In [ ]:
LATENT_DIM = 128
SEQ_LEN = 16
VOCAB_SIZE = 95
BATCH_SIZE = 96
EPOCHS = 60

In [ ]:
# Feature-wise Linear Modulation layer

class FiLM(layers.Layer):

    def __init__(self, units):
        super().__init__()
        self.gamma = layers.Dense(units)
        self.beta = layers.Dense(units)

    def call(self, x, cond):

        g = self.gamma(cond)
        b = self.beta(cond)

        return g * x + b

In [ ]:
def build_generator():

    noise = layers.Input(shape=(LATENT_DIM,))
    cond = layers.Input(shape=(1,))

    x = layers.Dense(256, activation="relu")(noise)

    film = FiLM(256)
    x = film(x, cond)

    x = layers.Dense(512, activation="relu")(x)

    film2 = FiLM(512)
    x = film2(x, cond)

    x = layers.Dense(SEQ_LEN * VOCAB_SIZE)(x)
    x = layers.Reshape((SEQ_LEN, VOCAB_SIZE))(x)

    output = layers.Softmax()(x)

    return tf.keras.Model([noise, cond], output)

In [ ]:
data = np.load("train_data.npy")

print("Training samples:", data.shape)

In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=2e-5,
    beta_1=0,
    beta_2=0.9
)

In [ ]:
generator = build_generator()

for epoch in range(EPOCHS):

    idx = np.random.randint(0, data.shape[0], BATCH_SIZE)
    real = data[idx]

    noise = np.random.normal(0,1,(BATCH_SIZE,LATENT_DIM))
    cond = np.random.randint(0,3,(BATCH_SIZE,1))

    with tf.GradientTape() as tape:

        fake = generator([noise,cond])
        loss = tf.reduce_mean(fake)

    grads = tape.gradient(loss, generator.trainable_variables)

    optimizer.apply_gradients(
        zip(grads, generator.trainable_variables)
    )

    print("Epoch:",epoch,"Loss:",float(loss))